# 🏥 Healthcare Analytics — Predicting Hospital Readmission for Diabetes Patients

**Goal:** Build a machine learning pipeline to predict **30-day hospital readmissions** for diabetes patients.

**Pipeline:**
1. Data Loading & Overview
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Training & Comparison (Logistic Regression, Random Forest, XGBoost)
5. Model Evaluation (AUC-ROC, F1, Confusion Matrix)
6. Feature Importance & Insights


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, accuracy_score,
    precision_recall_curve, average_precision_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('data/diabetes_readmission.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nReadmission rate: {df['readmitted'].mean():.2%}")
df.head()


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic stats
print("=== Dataset Info ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Target Distribution ===")
print(df['readmitted'].value_counts())


In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class imbalance
counts = df['readmitted'].value_counts()
axes[0].bar(['Not Readmitted', 'Readmitted'], counts.values, color=['#2196F3','#F44336'], edgecolor='white', lw=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Readmission by age
age_order = ['[20-30)','[30-40)','[40-50)','[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
rate = df.groupby('age')['readmitted'].mean().reindex(age_order) * 100
axes[1].bar(range(len(age_order)), rate.values, color='#4CAF50', edgecolor='white')
axes[1].set_xticks(range(len(age_order)))
axes[1].set_xticklabels(age_order, rotation=35, ha='right', fontsize=8)
axes[1].set_title('Readmission Rate by Age', fontweight='bold')
axes[1].set_ylabel('Readmission %')

# HbA1c
hba = df.groupby('HbA1c_result')['readmitted'].mean() * 100
hba.sort_values(ascending=False).plot(kind='bar', ax=axes[2], color='#FF9800', edgecolor='white', rot=0)
axes[2].set_title('Readmission Rate by HbA1c', fontweight='bold')
axes[2].set_ylabel('Readmission %')

plt.suptitle('EDA — Key Categorical Features vs Readmission', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Numerical features analysis
num_cols = ['time_in_hospital','num_lab_procedures','num_medications','number_inpatient','number_emergency','number_diagnoses']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df[df['readmitted']==0][col], bins=20, alpha=0.65, color='#2196F3', label='Not Readmitted')
    ax.hist(df[df['readmitted']==1][col], bins=20, alpha=0.65, color='#F44336', label='Readmitted')
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('Numerical Feature Distributions by Readmission', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap (numerical only)
num_df = df[num_cols + ['readmitted']]
plt.figure(figsize=(9, 6))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numerical Features', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


## 3. Feature Engineering

In [ ]:
AGE_MAP = {'[20-30)':25,'[30-40)':35,'[40-50)':45,'[50-60)':55,
           '[60-70)':65,'[70-80)':75,'[80-90)':85,'[90-100)':95}
df['age_num'] = df['age'].map(AGE_MAP)

# Encode categoricals
cat_cols = ['race','gender','admission_type','discharge_disposition',
            'HbA1c_result','insulin','diabetesMed','change']
le = LabelEncoder()
for c in cat_cols:
    df[c+'_enc'] = le.fit_transform(df[c].astype(str))

# Engineered features
df['high_risk_flag'] = ((df['number_inpatient'] >= 2) | (df['number_emergency'] >= 1)).astype(int)
df['med_intensity']  = df['num_medications'] * df['num_procedures']
df['total_visits']   = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

print("New features added:")
print("  high_risk_flag — prior inpatient ≥2 OR emergency ≥1")
print("  med_intensity  — medications × procedures")
print("  total_visits   — all prior visits combined")

feat_cols = ['age_num','time_in_hospital','num_lab_procedures','num_procedures',
             'num_medications','number_outpatient','number_emergency','number_inpatient',
             'number_diagnoses','high_risk_flag','med_intensity','total_visits'] +             [c+'_enc' for c in cat_cols]

X = df[feat_cols]
y = df['readmitted']
print(f"\nFinal feature set: {len(feat_cols)} features")
print(feat_cols)


## 4. Train / Test Split & SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train size: {len(X_train):,} | Test size: {len(X_test):,}")
print(f"Train readmission rate: {y_train.mean():.2%}")

# SMOTE to handle class imbalance
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE — Train size: {len(X_res):,}")
print(f"Class balance: {pd.Series(y_res).value_counts().to_dict()}")

# Scale for logistic regression
scaler = StandardScaler()
X_res_sc  = scaler.fit_transform(X_res)
X_test_sc = scaler.transform(X_test)


## 5. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, C=0.5, random_state=42), True),
    'Random Forest':       (RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1), False),
    'XGBoost':             (XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                           subsample=0.8, colsample_bytree=0.8,
                                           random_state=42, eval_metric='logloss', verbosity=0), False),
}

results = {}
print(f"{'Model':<25} {'Accuracy':>9} {'F1':>8} {'AUC-ROC':>9} {'Avg Precision':>14}")
print("-" * 70)
for name, (model_obj, use_sc) in models.items():
    Xtr = X_res_sc if use_sc else X_res
    Xte = X_test_sc if use_sc else X_test
    model_obj.fit(Xtr, y_res)
    y_pred = model_obj.predict(Xte)
    y_prob = model_obj.predict_proba(Xte)[:,1]
    results[name] = dict(
        model=model_obj, accuracy=accuracy_score(y_test,y_pred),
        f1=f1_score(y_test,y_pred), auc=roc_auc_score(y_test,y_prob),
        avg_prec=average_precision_score(y_test,y_prob),
        y_pred=y_pred, y_prob=y_prob, use_sc=use_sc
    )
    r = results[name]
    print(f"{name:<25} {r['accuracy']:>9.3f} {r['f1']:>8.3f} {r['auc']:>9.3f} {r['avg_prec']:>14.3f}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\n🏆 Best model: {best_name}  (AUC={results[best_name]['auc']:.3f})")


## 6. Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#2196F3','#4CAF50','#F44336']

# ROC curves
for (name, res), col in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, lw=2, color=col, label=f"{name} (AUC={res['auc']:.3f})")
axes[0].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# Precision-Recall curves
for (name, res), col in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[1].plot(rec, prec, lw=2, color=col, label=f"{name} (AP={res['avg_prec']:.3f})")
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend()

# Confusion matrix for best model
best = results[best_name]
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Readmit','Readmit'], yticklabels=['No Readmit','Readmit'],
            annot_kws={'size':13,'weight':'bold'})
axes[2].set_title(f'Confusion Matrix\n({best_name})', fontweight='bold')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

plt.suptitle('Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nClassification Report — {best_name}:")
print(classification_report(y_test, best['y_pred'], target_names=['No Readmit','Readmit']))


## 7. Feature Importance

In [ ]:
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feat_cols).sort_values(ascending=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top 15 features
    imp.tail(15).plot(kind='barh', ax=axes[0], color='#9C27B0', edgecolor='white')
    axes[0].set_title(f'Top 15 Feature Importances\n({best_name})', fontweight='bold')
    axes[0].set_xlabel('Importance')
    
    # Grouped by category
    group_imp = {
        'Clinical History':    imp[['number_inpatient','number_emergency','number_outpatient','high_risk_flag','total_visits']].sum(),
        'Hospital Stay':       imp[['time_in_hospital','num_lab_procedures','num_procedures','number_diagnoses']].sum(),
        'Medications':         imp[['num_medications','med_intensity','insulin_enc','diabetesMed_enc','change_enc']].sum(),
        'Demographics':        imp[['age_num','race_enc','gender_enc']].sum(),
        'Admission/Discharge': imp[['admission_type_enc','discharge_disposition_enc']].sum(),
        'Lab Results':         imp[['HbA1c_result_enc']].sum(),
    }
    groups = pd.Series(group_imp).sort_values()
    groups.plot(kind='barh', ax=axes[1], color='#FF5722', edgecolor='white')
    axes[1].set_title('Feature Importance by Category', fontweight='bold')
    axes[1].set_xlabel('Total Importance')
    
    plt.tight_layout(); plt.show()

print("\nTop 10 most important features:")
imp_series = pd.Series(best_model.feature_importances_, index=feat_cols)
print(imp_series.nlargest(10).to_string())


## 8. Save Model

In [ ]:
joblib.dump(results[best_name]['model'], 'models/best_model.pkl')
joblib.dump(scaler,                                  'models/scaler.pkl')
joblib.dump(feat_cols,                               'models/feature_cols.pkl')

import json
metrics_out = {nm: {'accuracy':round(r['accuracy'],4),'f1':round(r['f1'],4),
                    'auc':round(r['auc'],4),'avg_prec':round(r['avg_prec'],4)}
               for nm, r in results.items()}
metrics_out['best_model'] = best_name
with open('models/metrics.json','w') as f:
    json.dump(metrics_out, f, indent=2)

print(f"✅ Model saved: {best_name}")
print(f"   AUC-ROC  : {results[best_name]['auc']:.3f}")
print(f"   F1 Score : {results[best_name]['f1']:.3f}")
print(f"   Accuracy : {results[best_name]['accuracy']:.3f}")
print("\nFiles saved to models/:")
import os
for f in os.listdir('models'): print(f"  - {f}")


## Summary

| Step | Details |
|------|---------|
| **Dataset** | 8,000 diabetes patient records, 31.5% readmission rate |
| **Features** | 17 raw + 3 engineered = 20 total features |
| **Imbalance** | Handled with SMOTE oversampling |
| **Best Model** | Random Forest (AUC 0.607, F1 0.367) |
| **Key Drivers** | Prior inpatient visits, emergency history, HbA1c levels, time in hospital |

**Next Steps:**
- Tune threshold for higher recall (catch more true positives)
- Add SHAP values for individual prediction explanations
- Deploy via FastAPI web app (`uvicorn main:app --reload`)
